# ORSO Example: 2-Layer NF Inference

Smoke-test the 2-layer normalizing-flow checkpoint on `exp_data/ORSO_example.ort`.

## Setup

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib as mpl

mpl.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
import torch

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "exp_data" / "ORSO_example.ort").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the inverse-eval repo.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from device_utils import detect_torch_device, summarize_torch_backends
from nf_statistics import compute_nf_sample_statistics
from reflectorch import EasyInferenceModel


def to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def extend_prior_bounds_for_nf(prior_bounds):
    prior_bounds = list(prior_bounds)
    if len(prior_bounds) in (5, 8):
        return [*prior_bounds, (0.9, 1.1), (-10.0, -4.0)]
    if len(prior_bounds) in (7, 10):
        return prior_bounds
    raise ValueError(f"Unsupported NF prior_bounds length={len(prior_bounds)}")

plt.rcParams["text.usetex"] = False
torch.manual_seed(42)

print(f"Project root    : {PROJECT_ROOT}")
print(f"PyTorch version : {torch.__version__}")
print(f"Backends        : {summarize_torch_backends()}")

## Configuration

In [ ]:
ORSO_FILE = PROJECT_ROOT / "exp_data" / "ORSO_example.ort"
VENDOR_ROOT = PROJECT_ROOT / "vendor" / "nflows_reflectorch"

NF_CONFIG = "example_nf_config_reflectorch_L2.yaml"
NF_SAMPLES = 10000  # increase for final plots
DEVICE = detect_torch_device()
AMBIENT_SLD = None  # set to a value in 1e-6 A^-2 if the fronting medium is known

STRUCTURAL_PRIOR_BOUNDS = [
    (1.0, 750.0),
    (1.0, 750.0),
    (0.0, 60.0),
    (0.0, 60.0),
    (0.0, 60.0),
    (-8.0, 16.0),
    (-8.0, 16.0),
    (-8.0, 16.0),
]

PRIOR_BOUNDS = extend_prior_bounds_for_nf(STRUCTURAL_PRIOR_BOUNDS)

print(f"ORSO file      : {ORSO_FILE}")
print(f"NF config      : {NF_CONFIG}")
print(f"Device         : {DEVICE}")
print(f"Samples        : {NF_SAMPLES}")
print(f"Prior dim      : {len(PRIOR_BOUNDS)}")

## Load ORSO Data

In [ ]:
def load_orso_curve(path: Path):
    data = np.loadtxt(path, comments="#")
    if data.ndim != 2 or data.shape[1] < 3:
        raise ValueError(f"Expected at least Q, R, dR columns in {path}")

    q = data[:, 0]
    r = data[:, 1]
    dr = data[:, 2]
    dq = data[:, 3] if data.shape[1] >= 4 else None

    mask = np.isfinite(q) & np.isfinite(r) & np.isfinite(dr) & (q > 0) & (r > 0) & (dr > 0)
    if dq is not None:
        mask &= np.isfinite(dq) & (dq >= 0)
        dq = dq[mask]

    return q[mask], r[mask], dr[mask], dq


q_exp, r_exp, dr_exp, dq_exp = load_orso_curve(ORSO_FILE)

print(f"Loaded points : {len(q_exp)}")
print(f"Q range       : {q_exp.min():.5f} - {q_exp.max():.5f} A^-1")
print(f"R range       : {r_exp.min():.3e} - {r_exp.max():.3e}")
print(f"dR/R range    : {(dr_exp / r_exp).min():.3f} - {(dr_exp / r_exp).max():.3f}")
if dq_exp is not None:
    print(f"dQ/Q median   : {np.median(dq_exp / q_exp):.4f}")

assert len(q_exp) > 0
assert np.all(np.diff(q_exp) > 0)

## Run Inference

In [ ]:
model = EasyInferenceModel(
    config_name=NF_CONFIG,
    root_dir=str(VENDOR_ROOT),
    repo_id=None,
    device=DEVICE,
)

expected_dim = model.trainer.loader.prior_sampler.param_dim
assert len(PRIOR_BOUNDS) == expected_dim, (len(PRIOR_BOUNDS), expected_dim)

prediction = model.preprocess_and_sample(
    reflectivity_curve=r_exp,
    q_values=q_exp,
    sigmas=dr_exp,
    q_resolution=dq_exp if dq_exp is not None else 0.1,
    ambient_sld=AMBIENT_SLD,
    prior_bounds=PRIOR_BOUNDS,
    num_samples=NF_SAMPLES,
    sampling_batch_size=128,
    maximum_sim_batch_size=128,
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
    calc_log_likelihoods=True,
    enable_importance_sampling=True,
    clip_prediction=True,
)

print("Inference complete.")
print(f"Sampled params : {prediction['predicted_params_array'].shape}")
print(f"Sampled curves : {prediction['sampled_curves'].shape}")
print(f"Param names    : {prediction['param_names']}")

## MAP Sample

In [ ]:
log_likelihoods = to_numpy(prediction["log_likelihoods"])
samples = to_numpy(prediction["predicted_params_array"])
param_names = prediction["param_names"]
best_idx = int(np.argmax(log_likelihoods))
map_params = samples[best_idx]
stats = compute_nf_sample_statistics(samples, log_likelihoods)

print(f"MAP index       : {best_idx}")
print(f"MAP loglike     : {log_likelihoods[best_idx]:.4f}")
if "ess" in prediction:
    print(f"ESS             : {prediction['ess']:.1f} / {NF_SAMPLES}")
print()
print(f"{'Parameter':<22} {'Prior min':>10} {'MAP':>12} {'Mean':>12} {'Std':>12} {'Prior max':>10}")
print("-" * 84)
for i, name in enumerate(param_names):
    lo, hi = PRIOR_BOUNDS[i]
    print(
        f"{name:<22} {lo:>10.3f} {map_params[i]:>12.4f} "
        f"{stats['nf_params_mean'][i]:>12.4f} {stats['nf_params_std'][i]:>12.4f} {hi:>10.3f}"
    )

assert samples.shape == (NF_SAMPLES, expected_dim)
assert np.isfinite(map_params).all()

## Plot Fit and SLD

In [ ]:
q_plot = to_numpy(prediction["q_plot_pred"])
curve_map = to_numpy(prediction["sampled_curves"])[best_idx]
sld_x = to_numpy(prediction["sampled_sld_xaxis"])
sld_map = to_numpy(prediction["sampled_sld_profiles"])[best_idx]

fig, (ax_r, ax_sld) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

ax_r.errorbar(q_exp, r_exp, yerr=dr_exp, fmt=".", ms=3, alpha=0.45, label="ORSO")
ax_r.plot(q_plot, curve_map, lw=2, label="2-layer NF MAP")
ax_r.set(xlabel="Q (A^-1)", ylabel="Reflectivity", yscale="log")
ax_r.legend()

ax_sld.plot(sld_x, sld_map, lw=2)
ax_sld.set(xlabel="Depth (A)", ylabel="SLD (1e-6 A^-2)")

plt.show()

## NF Sampled SLD Profiles

Overlay every finite SLD profile sampled by the NF posterior. Multiple separated profile families for the same reflectivity curve indicate an ill-posed inverse result.

In [ ]:
def finite_sld_profiles(prediction):
    sld_x = to_numpy(prediction["sampled_sld_xaxis"])
    profiles = to_numpy(prediction["sampled_sld_profiles"])
    loglike = to_numpy(prediction["log_likelihoods"]).ravel()
    if profiles.ndim == 1:
        profiles = profiles[None, :]
    mask = np.isfinite(loglike) & np.all(np.isfinite(profiles), axis=1)
    profiles = profiles[mask]
    loglike = loglike[mask]
    if sld_x.ndim == 2:
        sld_x = sld_x[mask]
    if len(profiles) == 0:
        raise ValueError("No finite sampled SLD profiles available to plot.")
    return sld_x, profiles, loglike


sld_x_all, sld_profiles_all, sld_loglike_all = finite_sld_profiles(prediction)
best_sld_idx = int(np.argmax(sld_loglike_all))
order = np.argsort(sld_loglike_all)
line_alpha = min(0.18, max(0.01, 30 / len(order)))

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
for idx in order:
    x = sld_x_all[idx] if np.ndim(sld_x_all) == 2 else sld_x_all
    ax.plot(x, sld_profiles_all[idx], color="0.25", alpha=line_alpha, lw=0.8)

x_best = sld_x_all[best_sld_idx] if np.ndim(sld_x_all) == 2 else sld_x_all
ax.plot(x_best, sld_profiles_all[best_sld_idx], color="crimson", lw=2.4, label="MAP")
ax.set(xlabel="Depth (A)", ylabel="SLD (1e-6 A^-2)", title="ORSO NF sampled SLD profiles")
ax.legend()
plt.show()

print(f"Plotted {len(sld_profiles_all)} finite sampled SLD profiles.")
